In [ ]:
import os

os.chdir('/Users/sebert/Documents/BulkRNAseq_Liver')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
counts=pd.read_csv('GSE135251_norm_counts_TPM_GRCh38.p13_NCBI (1).tsv',index_col=0, sep='\t')
meta=pd.read_csv('GSE135251_series_matrix (2).txt',index_col=0, sep='\t')
genes=pd.read_csv('Human.GRCh38.p13.annot (1).tsv',index_col=0, sep='\t')

In [ ]:
counts_with_symbol = counts.copy()
counts_with_symbol.index = counts_with_symbol.index.map(lambda gene_id: genes.loc[gene_id, 'Symbol'] if gene_id in genes.index else gene_id)

In [ ]:
counts=counts_with_symbol

In [ ]:
counts.head()

In [ ]:
counts=counts.T

In [ ]:
counts.head()

In [ ]:
meta_with_group = meta.T.copy()
meta_with_group['Group'] = meta.loc['Group']
meta_with_group.head()

In [ ]:
merged_df = meta_with_group.merge(counts, left_index=True, right_index=True)

In [ ]:
merged_df.head()

In [ ]:
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 20
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

In [ ]:
from matplotlib.pyplot import rc_context

In [ ]:
merged_df = merged_df[merged_df['Group'] != 'control']

In [ ]:

from matplotlib import cm

# Generate 4 shades of magenta (light → intense) using "PuRd" colormap
magentas = [cm.PuRd(i) for i in np.linspace(0.3, 0.9, 5)]

# Add gray as the first color
custom_palette = magentas  # total = 5 colors

In [ ]:
GROUP_ORDER = ["NAFL", "NASH_F0-F1", "NASH_F2", "NASH_F3", "NASH_F4"]

ALL_GENES = ["CCL20", "CCL21", "CCL24", "CCL25", "CCL26", "CCL28", "CXCL9", "CXCL10",
             "CXCL11", "CXCL12", "CXCL13", "CXCL17", "XCL1", "PRDX1", "DDT"]
available_genes = [g for g in ALL_GENES if g in merged_df.columns]
missing = sorted(set(ALL_GENES) - set(available_genes))
if missing:
    print("Not found in count matrix:", missing)


plot_df = merged_df.melt(id_vars=["Group"], value_vars=available_genes,
                         var_name="Gene", value_name="TPM")
plot_df["TPM"] = pd.to_numeric(plot_df["TPM"])
plot_df["log1p_TPM"] = np.log1p(plot_df["TPM"])
plot_df = plot_df[plot_df["Group"].isin(GROUP_ORDER)].reset_index(drop=True)

print(plot_df.groupby("Group")["Gene"].count() / len(available_genes))  # n patients per group
plot_df.head()

In [ ]:
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
from statannotations.Annotator import Annotator

CONTROLS   = ["NAFL", "NASH_F0-F1"]
CORRECTION = "bonferroni"   
ALPHA      = 0.05
VALUE_COL  = "log1p_TPM"


def build_pairs(genes, controls=CONTROLS, groups=GROUP_ORDER):
    """Each control vs. every other group, per gene; each unordered pair only once."""
    pairs, seen = [], set()
    for gene in genes:
        for control in controls:
            for group in groups:
                if group == control:
                    continue
                key = (gene, frozenset((control, group)))
                if key not in seen:
                    seen.add(key)
                    pairs.append(((gene, control), (gene, group)))
    return pairs


def run_stats(df, pairs, method=CORRECTION, value_col=VALUE_COL):
    """Two-sided Mann–Whitney U per pair, then one correction across all pairs of the figure."""
    rows = []
    for (gene, g1), (_, g2) in pairs:
        x = df.loc[(df["Gene"] == gene) & (df["Group"] == g1), value_col]
        y = df.loc[(df["Gene"] == gene) & (df["Group"] == g2), value_col]
        res = mannwhitneyu(x, y, alternative="two-sided")
        rows.append({"Gene": gene, "Group1": g1, "Group2": g2,
                     "n1": len(x), "n2": len(y), "U": res.statistic, "p_raw": res.pvalue})
    stats_df = pd.DataFrame(rows)
    stats_df["p_adj"] = multipletests(stats_df["p_raw"], method=method)[1]
    stats_df["correction"] = method
    stats_df["n_tests"] = len(stats_df)
    stats_df["significant"] = stats_df["p_adj"] <= ALPHA
    return stats_df


def fmt_p(p):
    return f"$p$ = {p:.3f}" if p >= 0.001 else f"$p$ = {p:.1e}"


def plot_genes(df, genes, outfile, method=CORRECTION,
               title="Reanalysis of GSE135251: Selected Genes by Group", figsize=(11, 7)):
    genes = [g for g in genes if g in set(df["Gene"])]
    sub = df[df["Gene"].isin(genes)]

    pairs = build_pairs(genes)
    stats_df = run_stats(sub, pairs, method=method)
    assert len(pairs) == 7 * len(genes), len(pairs)
    print(f"{outfile}: {len(genes)} genes x 7 = {len(pairs)} unique comparisons, "
          f"correction = {method}, significant = {stats_df['significant'].sum()}")

    with rc_context({"font.size": 10}):
        fig, ax = plt.subplots(figsize=figsize)
        sns.boxplot(data=sub, x="Gene", y=VALUE_COL, hue="Group",
                    order=genes, hue_order=GROUP_ORDER, palette=custom_palette,
                    linewidth=0.5, showfliers=False, ax=ax)
        sns.stripplot(data=sub, x="Gene", y=VALUE_COL, hue="Group",
                      order=genes, hue_order=GROUP_ORDER, palette=custom_palette,
                      dodge=True, alpha=0.5, size=3, jitter=True, marker="o",
                      edgecolor="#808080", linewidth=0.6, legend=False, ax=ax)

        sig = stats_df[stats_df["significant"]]
        if len(sig):
            sig_pairs = [((r.Gene, r.Group1), (r.Gene, r.Group2)) for r in sig.itertuples()]
            annotator = Annotator(ax, sig_pairs, data=sub, x="Gene", y=VALUE_COL, hue="Group",
                                  order=genes, hue_order=GROUP_ORDER, verbose=False)
            annotator.configure(loc="inside", fontsize=12, line_width=0.5,
                                line_offset=5, verbose=0)
            annotator.set_custom_annotations([fmt_p(p) for p in sig["p_adj"]])
            annotator.annotate()

        ax.set_title(title, fontsize=12)
        ax.set_ylabel("log1p(TPM)", fontsize=12)
        ax.set_xlabel("Gene Symbol", fontsize=12)
        for lab in ax.get_xticklabels():
            lab.set_rotation(45); lab.set_ha("right")
            lab.set_fontstyle("italic"); lab.set_fontsize(12)

        handles, labels = ax.get_legend_handles_labels()
        ax.legend(handles[:len(GROUP_ORDER)], labels[:len(GROUP_ORDER)],
                  title=None, fontsize=12, bbox_to_anchor=(1, 1))

        fig.tight_layout()
        fig.savefig(outfile, bbox_inches="tight", transparent=True)
        plt.show()

    stats_df.to_csv(outfile.replace(".pdf", "_stats.csv"), index=False)
    return stats_df

In [ ]:

FIG3_GENES = ["CCL20", "CCL21", "CCL28", "CXCL9", "DDT"]
stats_fig3 = plot_genes(plot_df, FIG3_GENES, "GSE135251_Fig3_Boxplot_withStats.pdf")
stats_fig3

In [ ]:

SUPPL2_GENES = ["CCL20", "CCL21", "CCL24", "CCL25", "CCL26", "CCL28", "CXCL9", "CXCL10",
                "CXCL11", "CXCL12", "CXCL13", "CXCL17", "XCL1", "PRDX1"]
stats_s2 = plot_genes(plot_df, SUPPL2_GENES, "GSE135251_SupplFig2_Boxplot_withStats.pdf")
stats_s2[stats_s2["significant"]]